In [12]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.genmod.generalized_estimating_equations import GEE
from statsmodels.genmod.families import Gaussian, Binomial
from statsmodels.genmod.cov_struct import (
    Exchangeable,
    Independence,
    Autoregressive,
    Unstructured,
)

In [2]:
url = "https://gksmyth.github.io/ozdasl/oz/stroke.txt"
df = pd.read_csv(url, sep="\t")  # 区切りがタブっぽいので \t を指定

usecols = [
    "Subject",
    "Group",
    "Bart1",
    "Bart2",
    "Bart3",
    "Bart4",
    "Bart5",
    "Bart6",
    "Bart7",
    "Bart8",
]
df = df[usecols]

In [3]:
df.head()

,Subject,Group,Bart1,Bart2,Bart3,Bart4,Bart5,Bart6,Bart7,Bart8
0,I,E,45,45,45,45,80,80,80,90
1,II,E,20,25,25,25,30,35,30,50
2,III,E,50,50,55,70,70,75,90,90
3,IV,E,25,25,35,40,60,60,70,80
4,V,E,100,100,100,100,100,100,100,100


In [4]:
# Groupをダミー変数にする（alpha）
dummy = pd.get_dummies(df["Group"], columns=["Group"]).astype(int)
dummy = dummy.loc[np.repeat(dummy.index, 8)].reset_index(drop=True)

# 分散分析しやすいようにデータ加工
X = (
    df.drop(["Subject", "Group"], axis=1)
    .stack()
    .reset_index()
    .drop(["level_0"], axis=1)
    .rename(columns={"level_1": "week", 0: "Berthel_Score"})
)
X = pd.concat([dummy, X], axis=1)

In [5]:
X.head()

,E,F,G,week,Berthel_Score
0,1,0,0,Bart1,45
1,1,0,0,Bart2,45
2,1,0,0,Bart3,45
3,1,0,0,Bart4,45
4,1,0,0,Bart5,80


In [6]:
# Groupをダミー変数にする（alpha）
dummy = pd.get_dummies(df["Group"], columns=["Group"]).astype(int)
dummy = dummy.loc[np.repeat(dummy.index, 8)].reset_index(drop=True)

# 分散分析しやすいようにデータ加工
X = (
    df.drop(["Subject", "Group"], axis=1)
    .stack()
    .reset_index()
    .drop(["level_0"], axis=1)
    .rename(columns={"level_1": "week", 0: "Berthel_Score"})
)
X = pd.concat([dummy, X], axis=1)

# weekはそのまま数字を入れる（beta）
X["week"] = X["week"].str[4:].astype(int)
X["beta1_week"] = X["E"] * X["week"]
X["beta2_week"] = X["F"] * X["week"]
X["beta3_week"] = X["G"] * X["week"]

# 推定対象はalpha2, alpha3ではなく、alpha2 - alpha1, alpha3 - alpha1 とみなすので、
# 基準のとなるEはすべて１とする
# beta1, beta2, beta3 についても同様
X["E"] = 1
X["beta1_week"] = [1, 2, 3, 4, 5, 6, 7, 8] * 24

# 目的変数と説明変数に分ける
y = X["Berthel_Score"]
X["ids"] = np.repeat(np.arange(24), 8)
# X = X[["E", "F", "G", "beta1_week", "beta2_week", "beta3_week"]].values

In [7]:
X

,E,F,G,week,Berthel_Score,beta1_week,beta2_week,beta3_week,ids
0,1,0,0,1,45,1,0,0,0
1,1,0,0,2,45,2,0,0,0
2,1,0,0,3,45,3,0,0,0
3,1,0,0,4,45,4,0,0,0
4,1,0,0,5,80,5,0,0,0
...,...,...,...,...,...,...,...,...,...
187,1,0,1,4,35,4,0,4,23
188,1,0,1,5,40,5,0,5,23
189,1,0,1,6,50,6,0,6,23
190,1,0,1,7,65,7,0,7,23


## 独立の場合

In [8]:
# GEE（独立相関）
model = GEE(
    endog=y,
    exog=X[["E", "F", "G", "beta1_week", "beta2_week", "beta3_week"]].values,
    groups=X["ids"],
    family=Gaussian(),
    cov_struct=Independence(),
)
result = model.fit()

print(result.summary())
# 係数・標準誤差
print("params:\n", result.params)
print("bse:\n", result.bse)

                               GEE Regression Results                              
Dep. Variable:               Berthel_Score   No. Observations:                  192
Model:                                 GEE   No. clusters:                       24
Method:                        Generalized   Min. cluster size:                   8
                      Estimating Equations   Max. cluster size:                   8
Family:                           Gaussian   Mean cluster size:                 8.0
Dependence structure:         Independence   Num. iterations:                     2
Date:                     Sun, 24 Aug 2025   Scale:                         439.293
Covariance type:                    robust   Time:                         09:36:13
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         29.8214     10.176      2.931      0.003       9.877      49.765
x1     

## 交換可能（等相関）の場合

In [9]:
# GEE（独立相関）
model = GEE(
    endog=y,
    exog=X[["E", "F", "G", "beta1_week", "beta2_week", "beta3_week"]].values,
    groups=X["ids"],
    family=Gaussian(),
    cov_struct=Exchangeable(),
)
result = model.fit()

print(result.summary())
# 係数・標準誤差
print("params:\n", result.params)
print("bse:\n", result.bse)

                               GEE Regression Results                              
Dep. Variable:               Berthel_Score   No. Observations:                  192
Model:                                 GEE   No. clusters:                       24
Method:                        Generalized   Min. cluster size:                   8
                      Estimating Equations   Max. cluster size:                   8
Family:                           Gaussian   Mean cluster size:                 8.0
Dependence structure:         Exchangeable   Num. iterations:                     2
Date:                     Sun, 24 Aug 2025   Scale:                         439.293
Covariance type:                    robust   Time:                         09:36:13
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         29.8214     10.176      2.931      0.003       9.877      49.765
x1     

## 自己回帰の場合

In [11]:
# GEE（独立相関）
model = GEE(
    endog=y.values,
    exog=X[["E", "F", "G", "beta1_week", "beta2_week", "beta3_week"]].values,
    groups=X["ids"].values,
    time=X["week"].values,
    family=Gaussian(),
    cov_struct=Autoregressive(),
)
result = model.fit()

print(result.summary())
# 係数・標準誤差
print("params:\n", result.params)
print("bse:\n", result.bse)

/opt/anaconda3/envs/math_markething/lib/python3.12/site-packages/statsmodels/genmod/cov_struct.py:796: FutureWarning: grid=True will become default in a future version
  warnings.warn(


ValueError: Autoregressive: unable to find right bracket

## 無構造の場合

In [13]:
# GEE（独立相関）
model = GEE(
    endog=y.values,
    exog=X[["E", "F", "G", "beta1_week", "beta2_week", "beta3_week"]].values,
    groups=X["ids"].values,
    time=X["week"].values,
    family=Gaussian(),
    cov_struct=Unstructured(),
)
result = model.fit()

print(result.summary())
# 係数・標準誤差
print("params:\n", result.params)
print("bse:\n", result.bse)

                               GEE Regression Results                              
Dep. Variable:                           y   No. Observations:                  192
Model:                                 GEE   No. clusters:                       24
Method:                        Generalized   Min. cluster size:                   8
                      Estimating Equations   Max. cluster size:                   8
Family:                           Gaussian   Mean cluster size:                 8.0
Dependence structure:         Unstructured   Num. iterations:                    28
Date:                     Sun, 24 Aug 2025   Scale:                         478.232
Covariance type:                    robust   Time:                         09:43:10
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         30.4308     10.571      2.879      0.004       9.712      51.149
x1     

/opt/anaconda3/envs/math_markething/lib/python3.12/site-packages/statsmodels/genmod/cov_struct.py:301: RuntimeWarning: invalid value encountered in divide
  cov /= np.outer(sd, sd)


## 混合モデルの場合

In [23]:
from numpy.linalg import matrix_rank

In [32]:
# モデル: y ~ x1 + x2 + (1 | group)
model = sm.MixedLM.from_formula(
    "Berthel_Score ~ week",
    groups="ids",
    re_formula="1",  # ランダム効果の式
    data=X,
)
result = model.fit(reml=False)  # REML推定（reml=FalseならML推定）
print(result.summary())

           Mixed Linear Model Regression Results
Model:            MixedLM Dependent Variable: Berthel_Score
No. Observations: 192     Method:             ML           
No. Groups:       24      Scale:              79.8195      
Min. group size:  8       Log-Likelihood:     -736.7914    
Max. group size:  8       Converged:          Yes          
Mean group size:  8.0                                      
------------------------------------------------------------
             Coef.   Std.Err.    z     P>|z|  [0.025  0.975]
------------------------------------------------------------
Intercept    30.930     4.211   7.346  0.000  22.677  39.183
week          4.764     0.281  16.931  0.000   4.213   5.316
ids Var     377.031    13.368                               



In [33]:
model.exog.shape[1], matrix_rank(model.exog)

(2, 2)